In [1]:
from google.colab import files
import zipfile, os

# Upload zip
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

# Extract
extract_path = "/content/ipl_data"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files to:", extract_path)


Saving archive (4).zip to archive (4).zip
Extracted files to: /content/ipl_data


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib


In [6]:
matches_fp = os.path.join(extract_path, "matches.csv")
deliveries_fp = os.path.join(extract_path, "deliveries.csv")

matches = pd.read_csv(matches_fp)
deliveries = pd.read_csv(deliveries_fp)

# Merge
df = deliveries.merge(matches, left_on="match_id", right_on="id")

# Keep relevant columns
df = df[["match_id", "venue", "batting_team", "bowling_team", "total_runs",
         "over", "ball", "winner"]].dropna()

# Target: batting_team wins (1) or not (0)
df["target"] = np.where(df["winner"] == df["batting_team"], 1, 0)

# Runs remaining (approx calc)
df["cumulative_runs"] = df.groupby("match_id")["total_runs"].cumsum()
df["target_runs"] = df.groupby("match_id")["total_runs"].transform("max")
df["runs_remaining"] = df["target_runs"] - df["cumulative_runs"]

# Balls completed
df["balls_completed"] = ((df["over"] - 1) * 6 + df["ball"])

In [7]:
features = ["venue", "batting_team", "bowling_team", "runs_remaining", "balls_completed"]
X = df[features]
y = df["target"]

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", LogisticRegression(max_iter=200))
])

pipeline.fit(X_train, y_train)

# Evaluation
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]

print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("✅ ROC AUC:", roc_auc_score(y_test, y_prob))

# Save model
joblib.dump(pipeline, "/content/ipl_model.pkl")


✅ Accuracy: 0.6063454759106933
✅ ROC AUC: 0.6386152737834152


['/content/ipl_model.pkl']

In [8]:
!pip install gradio -q


In [12]:
import gradio as gr
import joblib
import pandas as pd

# Load trained model
model = joblib.load("/content/ipl_model.pkl")

# Teams aur Venues dataset se lelo
all_venues = sorted(df["venue"].dropna().unique().tolist())
all_teams = sorted(list(set(df["batting_team"].unique()) | set(df["bowling_team"].unique())))

def predict_probability(venue, batting_team, bowling_team, target, current_score, wickets, overs):
    balls_completed = int(overs * 6)
    runs_remaining = target - current_score

    input_df = pd.DataFrame({
        "venue": [venue],
        "batting_team": [batting_team],
        "bowling_team": [bowling_team],
        "runs_remaining": [runs_remaining],
        "balls_completed": [balls_completed]
    })

    # One-hot encoding
    input_df = pd.get_dummies(input_df)
    missing_cols = set(model.named_steps['scaler'].feature_names_in_) - set(input_df.columns)
    for col in missing_cols:
        input_df[col] = 0
    input_df = input_df[model.named_steps['scaler'].feature_names_in_]

    prob = model.predict_proba(input_df)[0][1]

    # 🏏 Output me team ka naam ke sath probability
    return {
        f"{batting_team} Win Probability": f"{prob*100:.2f}%",
        f"{bowling_team} Win Probability": f"{(1-prob)*100:.2f}%"
    }

# Gradio Interface
iface = gr.Interface(
    fn=predict_probability,
    inputs=[
        gr.Dropdown(all_venues, label="Venue"),
        gr.Dropdown(all_teams, label="Batting Team"),
        gr.Dropdown(all_teams, label="Bowling Team"),
        gr.Number(label="Target Score", value=150),
        gr.Number(label="Current Score", value=50),
        gr.Number(label="Wickets Fallen", value=2),
        gr.Number(label="Overs Completed", value=10.0)
    ],
    outputs="json",
    title="🏏 IPL Win Probability Predictor",
    description="Enter match details to get win probabilities."
)

iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ee8db230a10114461.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
